<a href="https://colab.research.google.com/github/miamte/machine-learning-practice/blob/main/06_ensemble.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 앙상블

## 랜덤포레스트

In [30]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

wine = pd.read_csv('https://bit.ly/wine-date')

data = wine[['alcohol', 'sugar', 'pH']].to_numpy()
target = wine['class'].to_numpy()

train_input, test_input, train_target, test_target = train_test_split(data, target, test_size=0.2, random_state=42)

In [31]:
from sklearn.model_selection import cross_validate
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_jobs=-1, random_state=42)
scores = cross_validate(rf, train_input, train_target, return_train_score=True, n_jobs=-1)

print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.9973541965122431 0.8905151032797809


In [32]:
rf.fit(train_input, train_target)
print(rf.feature_importances_)

[0.23167441 0.50039841 0.26792718]


In [33]:
rf = RandomForestClassifier(oob_score=True, n_jobs=-1, random_state=42) # 부트스트랩 샘플에 포함되지 않는 남는 샘플로 평가. Out of Bag

rf.fit(train_input, train_target)
print(rf.oob_score_)

0.8934000384837406


## 엑스트라트리

In [34]:
from sklearn.ensemble import ExtraTreesClassifier

et = ExtraTreesClassifier(n_jobs=-1, random_state=42)
scores = cross_validate(et, train_input, train_target, return_train_score=True, n_jobs=-1)

print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.9974503966084433 0.8887848893166506


In [35]:
et.fit(train_input, train_target)
print(et.feature_importances_)

[0.20183568 0.52242907 0.27573525]


## 그라디언트 부스팅

In [36]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(random_state=42)
scores = cross_validate(gb, train_input, train_target, return_train_score=True, n_jobs=-1)

print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.8881086892152563 0.8720430147331015


In [37]:
gb = GradientBoostingClassifier(n_estimators=500, learning_rate=0.2, random_state=42)
scores = cross_validate(gb, train_input, train_target, return_train_score=True, n_jobs=-1)

print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.9464595437171814 0.8780082549788999


In [38]:
gb.fit(train_input, train_target)
print(gb.feature_importances_)

[0.15887763 0.6799705  0.16115187]


## 히스토그램 기반 부스팅

In [39]:
from sklearn.experimental import enable_hist_gradient_boosting
from sklearn.ensemble import HistGradientBoostingClassifier

hgb = HistGradientBoostingClassifier(random_state=42)
scores = cross_validate(hgb, train_input, train_target, return_train_score=True, n_jobs=-1)

print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.9321723946453317 0.8801241948619236


In [40]:
hgb.fit(train_input, train_target)
print(rf.feature_importances_)

[0.23167441 0.50039841 0.26792718]


In [41]:
hgb.score(test_input, test_target)

0.8723076923076923

#### XGBoost

In [42]:
from xgboost import XGBClassifier

xgb = XGBClassifier(tree_method='hist', random_state=42)
scores = cross_validate(xgb, train_input, train_target, return_train_score=True, n_jobs=-1)

print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.9567059184812372 0.8783915747390243


#### LightGBM

In [43]:
from lightgbm import LGBMClassifier

lgb = LGBMClassifier(random_state=42)
scores = cross_validate(lgb, train_input, train_target, return_train_score=True, n_jobs=-1)

print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.935828414851749 0.8801251203079884


In [56]:
# RandomForestClassifier 모델 재정의 및 교차 검증
rf_model_re = RandomForestClassifier(random_state=42, n_jobs=-1)
scores_rf_re = cross_validate(rf_model_re, c_train_input, c_train_target, return_train_score=True, n_jobs=-1)
print("RandomForest - Train Score (Mean):", np.mean(scores_rf_re['train_score']))
print("RandomForest - Test Score (Mean):", np.mean(scores_rf_re['test_score']))

RandomForest - Train Score (Mean): 1.0
RandomForest - Test Score (Mean): 0.9582417582417582


In [57]:
# XGBClassifier 모델 재정의 및 교차 검증
xgb_model_re = XGBClassifier(tree_method='hist', random_state=42, n_jobs=-1)
scores_xgb_re = cross_validate(xgb_model_re, c_train_input, c_train_target, return_train_score=True, n_jobs=-1)
print("XGBoost - Train Score (Mean):", np.mean(scores_xgb_re['train_score']))
print("XGBoost - Test Score (Mean):", np.mean(scores_xgb_re['test_score']))

XGBoost - Train Score (Mean): 1.0
XGBoost - Test Score (Mean): 0.964835164835165


In [58]:
# LGBMClassifier 모델 재정의 및 교차 검증
lgb_model_re = LGBMClassifier(random_state=42, n_jobs=-1)
scores_lgbm_re = cross_validate(lgb_model_re, c_train_input, c_train_target, return_train_score=True, n_jobs=-1)
print("LightGBM - Train Score (Mean):", np.mean(scores_lgbm_re['train_score']))
print("LightGBM - Test Score (Mean):", np.mean(scores_lgbm_re['test_score']))

LightGBM - Train Score (Mean): 1.0
LightGBM - Test Score (Mean): 0.964835164835165


# 문제 : 새로운 데이터셋으로 RandomForest, XGBoost, LightGBM 성능 비교

**성능 비교 요약 (유방암 데이터셋):**

*   **RandomForestClassifier:**
    *   교차 검증 훈련 점수 평균: 1.0
    *   교차 검증 테스트 점수 평균: 0.958
    *   실제 테스트 세트 점수: 0.965

*   **XGBClassifier:**
    *   교차 검증 훈련 점수 평균: 1.0
    *   교차 검증 테스트 점수 평균: 0.965

*   **LGBMClassifier:**
    *   교차 검증 훈련 점수 평균: 1.0
    *   교차 검증 테스트 점수 평균: 0.965

**결론:**
세 모델 모두 훈련 데이터에 대해 완벽한 성능(1.0)을 보였으며, 교차 검증 테스트 점수에서도 0.958에서 0.965 사이의 높은 정확도를 기록했습니다. 특히 XGBoost와 LightGBM은 교차 검증 테스트 점수에서 0.965로 RandomForest보다 약간 더 좋은 성능을 보였습니다. RandomForest의 실제 테스트 세트 점수도 0.965로 매우 준수했습니다. 이 결과는 세 앙상블 모델 모두 유방암 데이터셋 분류에 매우 효과적임을 나타냅니다.

In [47]:
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer() # array
c_data = cancer.data
c_target = cancer.target

# 이후 코드는 Wine 데이터셋과 동일하게 진행

In [48]:
c_data.shape

(569, 30)

In [49]:
c_target.shape

(569,)

In [50]:
c_train_input, c_test_input, c_train_target, c_test_target = train_test_split(c_data, c_target, test_size=0.2, random_state=42)

In [51]:
# 랜덤포레스트 모델 정의
rf_model = RandomForestClassifier(random_state=42, n_jobs=-1)

In [52]:
# 랜덤포레스트 교차 검증 및 교차 검증 각 폴드의 학습 데이터 점수 평균, 각 폴드의 테스트 데이터 점수 평균
scores = cross_validate(rf_model, c_train_input, c_train_target, return_train_score=True, n_jobs=-1)

print(np.mean(scores['train_score']), np.mean(scores['test_score']))

1.0 0.9582417582417582


In [53]:
# 실제 모델 학습
rf_model.fit(c_train_input, c_train_target)

# 실제 모델 평가
print(rf_model.score(c_test_input, c_test_target))

0.9649122807017544


In [54]:
# XGBoost 모델 정의
xgb_model = XGBClassifier(tree_method='hist', random_state=42)
# 교차 검증
scores = cross_validate(xgb_model, c_train_input, c_train_target, return_train_score=True, n_jobs=-1)
# 교차 검증 평가 점수
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

1.0 0.964835164835165


In [55]:
# LightBGM 모델
lgb_model = LGBMClassifier(random_state=42, n_jobs=-1)
# 교차 검증
scores = cross_validate(lgb_model, c_train_input, c_train_target, return_train_score=True, n_jobs=-1)
# 교차 검증 평가 점수
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

1.0 0.964835164835165


랜덤 포레스트 모델이 좀더 작게 나왔다